### Etapa 1 — Configuração e ingestão Bronze

Cria a sessão do Spark, lê o CSV de clientes da Olist e infere automaticamente os tipos das colunas. Ao final, exibe uma amostra dos dados e o schema carregado.

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Initialize SparkSession
spark = SparkSession.builder \
    .appName("CSV to DataFrame") \
    .getOrCreate()

# 2. Read CSV file into a DataFrame
df = spark.read.csv(
    "csv/olist_customers_dataset.csv", 
    header=True, 
    inferSchema=True
)

# 3. Inspect the data and schema
df.show(5)
df.printSchema()

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows

root
 |-- customer_id: string 

### Etapa 2 — Persistência na camada Bronze

Define os diretórios das camadas e grava o DataFrame original no formato Parquet na pasta `clientes_base/bronze`. Em seguida, realiza uma nova leitura dessa camada para confirmar a persistência dos dados.

In [3]:
bronze_path = "./clientes_base/bronze"
gold_path = "./clientes_base/gold"

os.makedirs(bronze_path, exist_ok=True)
os.makedirs(gold_path, exist_ok=True)

df.write.mode("overwrite").parquet(bronze_path)
df_bronze = spark.read.parquet(bronze_path)

print(f"Data successfully written to {bronze_path}")

Data successfully written to ./clientes_base/bronze


### Etapa 3 — Preparação da camada Silver

Particiona os dados pelo identificador único do cliente e conta quantos registros existem para cada `customer_unique_id`. A contagem é armazenada na coluna `count_unique_id` e a base enriquecida é persistida em Parquet.

In [4]:
silver_path = "./clientes_base/silver/count_unique_id"

os.makedirs(silver_path, exist_ok=True)

janela = Window.partitionBy("customer_unique_id")
df_filtered = df_bronze.withColumn("count_unique_id", F.count("customer_unique_id").over(janela))
df_filtered.show()
df_total_unique_id = df_filtered.count()
print(f"Total unique_id rows: {df_total_unique_id}")

df_filtered.write.mode("overwrite").parquet(silver_path)
df_silver_total = spark.read.parquet(silver_path)

print(f"Data successfully written to {silver_path}")

+--------------------+--------------------+------------------------+-----------------+--------------+---------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|    customer_city|customer_state|count_unique_id|
+--------------------+--------------------+------------------------+-----------------+--------------+---------------+
|2f29573c8cac5a7be...|0006fdc98a402fceb...|                   29400|    mimoso do sul|            ES|              1|
|40f0183f7439212e8...|00090324bbad0e934...|                   13054|         campinas|            SP|              1|
|0e114b02a45c98760...|000c8bdb58a29e711...|                   31555|   belo horizonte|            MG|              1|
|064064dd94c430137...|00115fc7123b5310c...|                   71015|         brasilia|            DF|              1|
|c267e32fea07ffc4d...|001f3c4211216384d...|                    8240|        sao paulo|            SP|              1|
|063a5ea8e53ff62d2...|0023557a94bef0038...|             

### Etapa 4 — Quarentena de registros duplicados

Filtra os registros cujo `customer_unique_id` aparece mais de uma vez. Esses dados são separados na zona de quarentena da camada Silver para análise posterior, sem serem utilizados ainda na agregação final.

In [5]:
silver_path_quarentine = "./clientes_base/silver/count_unique_id_duplicated"

os.makedirs(silver_path_quarentine, exist_ok=True)


df_duplicados = df_filtered.filter(F.col("count_unique_id") > 1)
df_duplicados.show()
df_total_duplicated = df_duplicados.count()
print(f"Total duplicated rows: {df_total_duplicated}")

df_duplicados.write.mode("overwrite").parquet(silver_path_quarentine)
df_silver_quarentine = spark.read.parquet(silver_path_quarentine)

print(f"Data successfully written to {silver_path_quarentine}")

+--------------------+--------------------+------------------------+--------------+--------------+---------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix| customer_city|customer_state|count_unique_id|
+--------------------+--------------------+------------------------+--------------+--------------+---------------+
|1b4a75b3478138e99...|004288347e5e88a27...|                   26220|   nova iguacu|            RJ|              2|
|f6efe5d5c7b85e123...|004288347e5e88a27...|                   26220|   nova iguacu|            RJ|              2|
|49cf243e0d353cd41...|004b45ec5c6418746...|                   57055|        maceio|            AL|              2|
|d95f60d70d9ea9a7f...|004b45ec5c6418746...|                   57035|        maceio|            AL|              2|
|450a869eb52446baa...|013ef03e0f3f408dd...|                   13092|      campinas|            SP|              2|
|d4f243fba8f7e8213...|013ef03e0f3f408dd...|                   13092|      campin

### Etapa 5 — Seleção dos registros não duplicados

Mantém apenas os clientes que possuem um único registro na base. O conjunto limpo é salvo separadamente na camada Silver e será usado como entrada da camada Gold.

In [6]:
silver_path_nduplicated = "./clientes_base/silver/count_unique_id_nduplicated"

os.makedirs(silver_path_nduplicated, exist_ok=True)


df_clean = df_filtered.filter(F.col("count_unique_id") == 1)
df_clean.show()
df_total_clean = df_clean.count()
print(f"Total clean rows: {df_total_clean}")

df_clean.write.mode("overwrite").parquet(silver_path_nduplicated)
df_silver_clean = spark.read.parquet(silver_path_nduplicated)

print(f"Data successfully written to {silver_path_nduplicated}")

+--------------------+--------------------+------------------------+-----------------+--------------+---------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|    customer_city|customer_state|count_unique_id|
+--------------------+--------------------+------------------------+-----------------+--------------+---------------+
|2f29573c8cac5a7be...|0006fdc98a402fceb...|                   29400|    mimoso do sul|            ES|              1|
|40f0183f7439212e8...|00090324bbad0e934...|                   13054|         campinas|            SP|              1|
|0e114b02a45c98760...|000c8bdb58a29e711...|                   31555|   belo horizonte|            MG|              1|
|064064dd94c430137...|00115fc7123b5310c...|                   71015|         brasilia|            DF|              1|
|c267e32fea07ffc4d...|001f3c4211216384d...|                    8240|        sao paulo|            SP|              1|
|063a5ea8e53ff62d2...|0023557a94bef0038...|             

### Etapa 6 — Agregação na camada Gold

Agrupa os registros limpos por `customer_state` e conta quantos clientes existem em cada unidade federativa. O resultado consolidado é gravado em Parquet na camada Gold.

In [7]:
gold_path = "./clientes_base/gold/custumer_state_count"

os.makedirs(gold_path, exist_ok=True)

df_gold_state = (df_silver_clean.groupby("customer_state").agg(F.count("customer_state").alias("customer_state_count")))
df_gold_state.show()

df_gold_state.write.mode("overwrite").parquet(gold_path)
df_gold_state = spark.read.parquet(gold_path)

print(f"Data successfully written to {gold_path}")

+--------------+--------------------+
|customer_state|customer_state_count|
+--------------+--------------------+
|            SC|                3433|
|            RO|                 229|
|            PI|                 471|
|            AM|                 140|
|            RR|                  44|
|            GO|                1886|
|            TO|                 266|
|            MT|                 845|
|            SP|               38989|
|            PB|                 504|
|            ES|                1906|
|            RS|                5108|
|            MS|                 675|
|            AL|                 388|
|            MG|               10911|
|            PA|                 925|
|            BA|                3182|
|            SE|                 331|
|            PE|                1569|
|            CE|                1289|
+--------------+--------------------+
only showing top 20 rows

Data successfully written to ./clientes_base/gold/custumer_sta